# Consumer: Text Inference

Simulates an external consumer that:
1. Authenticates with MinIO and downloads the latest production model's
   `feature_names.txt` and `scaler.pkl`
2. Calls `/embed/text` with `{text, tags}` — the inference service is
   stateless; the caller owns the vocabulary
3. Scales the returned 0/1 vector locally using the downloaded scaler
4. Optionally downloads `ltn.h5` and runs prediction locally

## 1. Configuration

**Option A:** Upload a `.env` file — `MINIO_USER` and `MINIO_PASS` will be populated automatically.
**Option B:** Fill in the values manually in the cell below.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

MINIO_USER = "your_minio_username"
MINIO_PASS = "your_minio_password"


def _parse_env(content: str) -> dict:
    result = {}
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            key, _, value = line.partition("=")
            result[key.strip()] = value.strip().strip('"').strip("'")
    return result


def _on_env_upload(change):
    global MINIO_USER, MINIO_PASS
    value = change["new"]
    entries = value if isinstance(value, (list, tuple)) else value.values()
    for entry in entries:
        raw = entry["content"]
        text = bytes(raw).decode("utf-8") if not isinstance(raw, str) else raw
        env = _parse_env(text)
        if "MINIO_USER" in env:
            MINIO_USER = env["MINIO_USER"]
        if "MINIO_PASS" in env:
            MINIO_PASS = env["MINIO_PASS"]
        name = entry.get("name", entry.get("metadata", {}).get("name", ".env"))
        env_status.value = (
            f"<b style='color:green'>Loaded {name}</b> — "
            f"MINIO_USER=<code>{MINIO_USER}</code>, "
            f"MINIO_PASS=<code>{'*' * len(MINIO_PASS)}</code>"
        )


uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
env_status = widgets.HTML(value="<i style='color:grey'>No .env file uploaded yet.</i>")
uploader.observe(_on_env_upload, names="value")
display(widgets.VBox([uploader, env_status]))

In [ ]:
import os

# --- Manual config (skip if you uploaded a .env above) ---
# MINIO_USER = "your_minio_username"
# MINIO_PASS = "your_minio_password"

MINIO_BUCKET  = "smart-healthcare-diabetes-models"
INFERENCE_URL = os.getenv("INFERENCE_API_URL", "http://localhost:8890")

print(f"Inference API : {INFERENCE_URL}")
print(f"MinIO bucket  : {MINIO_BUCKET}")

## 2. MinIO helpers (self-contained)

In [ ]:
import requests

MINIO_HOST = "https://humaine-minio-api.euprojects.net"


def minio_auth(user, password):
    resp = requests.post(
        f"{MINIO_HOST}/auth/auth",
        data={"username": user, "password": password},
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    if resp.status_code == 200:
        return resp.json()["access_token"]
    raise RuntimeError(f"MinIO auth failed: {resp.status_code} {resp.text}")


def minio_read_json(token, bucket, object_name):
    resp = requests.get(
        f"{MINIO_HOST}/main_ops/download/{bucket}/{object_name}",
        headers={"Authorization": f"Bearer {token}", "accept": "application/json"},
    )
    return resp.json() if resp.status_code == 200 else None


def minio_download(token, bucket, object_name, file_path):
    resp = requests.get(
        f"{MINIO_HOST}/main_ops/download/{bucket}/{object_name}",
        headers={"Authorization": f"Bearer {token}", "accept": "application/json"},
    )
    if resp.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(resp.content)
    else:
        raise RuntimeError(f"Download failed ({object_name}): {resp.status_code} {resp.text}")

## 3. Authenticate with MinIO

In [ ]:
token = minio_auth(MINIO_USER, MINIO_PASS)
print("Authenticated successfully.")

## 4. Locate the latest production **text** model

Text and image models are published under type-specific subdirectories
(`published-models/{version}/text/...` and `.../images/...`). We filter the
index for `data_type == "text"` and `status == "production"` and pick the
most recent entry.

In [ ]:
DATA_TYPE   = "text"
DATA_SUBDIR = "text"

index = minio_read_json(token, MINIO_BUCKET, "published-models/index.json") or []
promoted = [
    e for e in index
    if e.get("status") == "production" and e.get("data_type") == DATA_TYPE
]
if not promoted:
    raise RuntimeError(f"No production-promoted {DATA_TYPE} model found in published-models/index.json.")

production_version = promoted[0]["version"]
prefix = f"published-models/{production_version}/{DATA_SUBDIR}"
print(f"Using production version: {production_version}")
print(f"Artifact prefix        : {prefix}")

## 5. Download feature_names.txt and scaler.pkl from MinIO

For text models, every entry in `feature_names.txt` is a tag —
there are no numeric columns.

In [ ]:
import joblib

local_features_path = f"./downloaded_feature_names_{production_version}.txt"
local_scaler_path   = f"./downloaded_scaler_{production_version}.pkl"

minio_download(token, MINIO_BUCKET, f"{prefix}/feature_names.txt", local_features_path)
minio_download(token, MINIO_BUCKET, f"{prefix}/scaler.pkl",        local_scaler_path)

with open(local_features_path) as f:
    feature_names = [ln.strip() for ln in f if ln.strip()]

scaler = joblib.load(local_scaler_path)
tags   = feature_names  # text: every feature column is a tag

print(f"N features: {len(feature_names)}")
print("Sample tags:")
for t in tags[:10]:
    print(f"  {t}")

## 6. Health check

In [ ]:
resp = requests.get(f"{INFERENCE_URL}/health")
resp.raise_for_status()
print(resp.json())

## 7. Embed a text sample

POST `{text, tags}` — the response `features` list is ordered to match the
`tags` list we sent, i.e. `feature_names` order.

In [ ]:
TEXT_SAMPLE = "WINNER!! You have been selected for a free prize. Call now to claim!"

resp = requests.post(
    f"{INFERENCE_URL}/embed/text",
    json={"text": TEXT_SAMPLE, "tags": tags},
)
resp.raise_for_status()
embed = resp.json()

assert embed["tags"] == tags, "Service must echo tags in the order they were sent."
print(f"Received {embed['n_features']} raw 0/1 features.")

## 8. Scale the vector locally

In [ ]:
import numpy as np

raw_vector    = np.array(embed["features"], dtype=np.float32).reshape(1, -1)
scaled_vector = scaler.transform(raw_vector).astype(np.float32).flatten()

print("Feature vector (post-scaler):")
for name, raw, scaled in zip(feature_names, raw_vector.flatten(), scaled_vector):
    bar = 'X' * int(abs(scaled) * 20)
    print(f"  {name:<30s} raw={int(raw)}  scaled={scaled:+.4f}  {bar}")

## 9. Visualise

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, max(4, len(feature_names) * 0.35)))
colors = ['steelblue' if v > 0 else 'lightgrey' for v in scaled_vector]
ax.barh(feature_names, scaled_vector, color=colors)
ax.set_xlabel('Feature value (scaler-transformed)')
ax.set_title('Text embedding — feature vector')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 10. (Optional) Download the model and predict locally

The inference service no longer exposes `/predict`. Consumers that need a
score download `ltn.h5` themselves and call Keras directly.

In [ ]:
import tensorflow as tf

local_model_path = f"./downloaded_ltn_{production_version}.h5"
minio_download(token, MINIO_BUCKET, f"{prefix}/ltn.h5", local_model_path)

model = tf.keras.models.load_model(local_model_path)
score = float(np.squeeze(model.predict(scaled_vector.reshape(1, -1), verbose=0)))
label = int(score > 0.5)

print(f"Score : {score:.4f}")
print(f"Label : {label}  ({'positive' if label == 1 else 'negative'} class)")

## 11. Batch embedding

In [ ]:
import pandas as pd

texts = [
    "WINNER!! You have been selected for a free prize. Call now to claim!",
    "Hey, are you coming to the meeting tomorrow at 10am?",
    "Urgent! Your account has been compromised. Click here to verify.",
]

rows = []
for text in texts:
    r = requests.post(
        f"{INFERENCE_URL}/embed/text",
        json={"text": text, "tags": tags},
    )
    r.raise_for_status()
    rows.append(r.json()["features"])

raw_matrix    = np.array(rows, dtype=np.float32)
scaled_matrix = scaler.transform(raw_matrix)

df = pd.DataFrame(scaled_matrix, columns=feature_names)
df.index = [f"sample_{i}" for i in range(len(texts))]
df

## 12. Cleanup (optional)

In [ ]:
for path in [local_features_path, local_scaler_path, locals().get('local_model_path')]:
    if path and os.path.exists(path):
        os.remove(path)
        print(f"Removed: {path}")